In [ ]:
!pip install rapidfuzz duckdb polars sentence-transformers faiss-gpu

In [1]:
# ============================================================
# Cell 1 — Imports
# ============================================================

import os
import re
import gc
import ast
import json
import math
import time
import sqlite3
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import joblib

from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score

try:
    from unidecode import unidecode
except Exception:
    unidecode = None

# Optional GPU / semantic retrieval stack.
try:
    import torch
except Exception:
    torch = None

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

try:
    import faiss
except Exception:
    faiss = None

print("Imports completed.")


Imports completed.


## 1. CUDA connection check

### Why this cell exists

Your RTX 3050 has 4 GB VRAM. The full 1–2 GB dataset must remain on disk / CPU RAM.

The GPU will only be used for **small batched multilingual embedding inference** after blocking.

We will therefore verify CUDA first and never move complete TSV files to VRAM.

In [2]:
# ============================================================
# Cell 2 — CUDA / GPU connection check
# ============================================================

if torch is None:
    print("PyTorch is not installed.")
else:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA runtime:", torch.version.cuda)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

        total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"Total GPU VRAM: {total_vram_gb:.2f} GB")

        # Leave headroom for the driver and CUDA runtime.
        # This is an allocator limit, not a substitute for small batches.
        try:
            torch.cuda.set_per_process_memory_fraction(0.78, device=0)
            print("PyTorch allocator cap: ~78% of GPU VRAM")
        except Exception as e:
            print("Could not set allocator cap:", e)
    else:
        print("WARNING: CUDA is unavailable. Notebook will use CPU fallback.")


PyTorch: 2.5.1+cu121
CUDA available: True
Torch CUDA runtime: 12.1
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total GPU VRAM: 4.00 GB
PyTorch allocator cap: ~78% of GPU VRAM


## 2. Local paths and constants

### Path convention

Keep `root_path` constant throughout the notebook, as in your original notebook.

The expected structure is:

```text
root_path/
└── student_resource/
    ├── dataset/
    │   ├── train/
    │   └── test/
    ├── utils/
    └── output/
```

The output directory will be created automatically.

In [3]:
# ============================================================
# Cell 3 — Paths / constants
# ============================================================

root_path = r"C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource"

train_dir = os.path.join(root_path, "student_resource", "dataset", "train")
test_dir = os.path.join(root_path, "student_resource", "dataset", "test")
utils_dir = os.path.join(root_path, "student_resource", "utils")

artifact_dir = os.path.join(root_path, "student_resource", "artifacts")
output_dir = os.path.join(root_path, "student_resource", "output")

ground_truth_file = os.path.join(train_dir, "train_ground_truth.tsv")

train_source1_file = os.path.join(train_dir, "train_source1.tsv")
train_source2_file = os.path.join(train_dir, "train_source2.tsv")
train_source3_file = os.path.join(train_dir, "train_source3.tsv")

test_source1_file = os.path.join(test_dir, "test_source1.tsv")
test_source2_file = os.path.join(test_dir, "test_source2.tsv")
test_source3_file = os.path.join(test_dir, "test_source3.tsv")

validator_file = os.path.join(utils_dir, "validate_submission.py")

# Local pretrained multilingual model.
# IMPORTANT: do not point this to an online model name if the challenge
# interprets "no external data" strictly. Use a permitted local model directory.
embedding_model_path = r"C:\models\multilingual-e5-small"

# 65 / 20 / 15 split.
TRAIN_FRAC = 0.65
VAL_FRAC = 0.20
TEST_FRAC = 0.15
RANDOM_SEED = 42

# RTX 3050 4 GB: start conservatively.
GPU_EMBED_BATCH_SIZE = 8
ANN_K = 50

# Candidate generation safeguards.
MAX_BLOCK_SIZE = 5000
MAX_BLOCK_KEYS_PER_ENTITY = 8

# Training sample size to keep CPU RAM manageable.
MAX_TRAIN_PAIRS_PER_SOURCE = 250_000
HARD_NEGATIVES_PER_ENTITY = 8

print("root_path:", root_path)
print("train_dir:", train_dir)
print("test_dir:", test_dir)
print("artifact_dir:", artifact_dir)
print("output_dir:", output_dir)


root_path: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource
train_dir: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\dataset\train
test_dir: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\dataset\test
artifact_dir: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts
output_dir: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\output


### Check that the local files exist

Before any large operation, verify the folder and the exact TSV names.

In [4]:
# ============================================================
# Cell 3A — Path existence check
# ============================================================

required_paths = {
    "root_path": root_path,
    "train_dir": train_dir,
    "test_dir": test_dir,
    "ground_truth_file": ground_truth_file,
    "train_source1_file": train_source1_file,
    "train_source2_file": train_source2_file,
    "train_source3_file": train_source3_file,
    "test_source1_file": test_source1_file,
    "test_source2_file": test_source2_file,
    "test_source3_file": test_source3_file,
    "validator_file": validator_file,
}

for name, path in required_paths.items():
    print(f"{name:24s} -> {'OK' if os.path.exists(path) else 'MISSING'}")


root_path                -> OK
train_dir                -> OK
test_dir                 -> OK
ground_truth_file        -> OK
train_source1_file       -> OK
train_source2_file       -> OK
train_source3_file       -> OK
test_source1_file        -> OK
test_source2_file        -> OK
test_source3_file        -> OK
validator_file           -> OK


### Check that the TSV files can be parsed with Polars

This is the same idea as your original notebook: use `scan_csv()` and collect only a small sample.

`scan_csv()` is lazy; it does **not** load the complete 1–2 GB file into RAM.

In [5]:
# ============================================================
# Cell 3B — Polars lazy parsing smoke test
# ============================================================

sample_train_files = [
    "train_ground_truth.tsv",
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
]

sample_test_files = [
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
]

def show_tsv_sample(path, n=5):
    lf = pl.scan_csv(
        path,
        separator="\t",
        has_header=True,
        infer_schema=False,
        ignore_errors=False,
    )
    return lf.head(n).collect()

print("=== TRAIN ===")
for fn in sample_train_files:
    p = os.path.join(train_dir, fn)
    print(f"\n{fn}")
    try:
        display(show_tsv_sample(p, 5))
    except Exception as e:
        print("ERROR:", e)

print("\n=== TEST ===")
for fn in sample_test_files:
    p = os.path.join(test_dir, fn)
    print(f"\n{fn}")
    try:
        display(show_tsv_sample(p, 5))
    except Exception as e:
        print("ERROR:", e)


=== TRAIN ===

train_ground_truth.tsv


source1_entity_id,matched_entity_ids
str,str
"""S1-965667""","""S2-681193310,S2-743505751,S3-7…"
"""S1-55344266""","""S2-249013014,S2-197070651,S3-4…"
"""S1-343815751""","""S2-790675320,S2-479876582,S3-8…"
"""S1-656753428""","""S2-153058913,S2-24659151,S3-67…"
"""S1-102811957""","""S2-478959098,S2-553508714,S2-6…"



train_source1.tsv


entity_id,business_name,business_address,country
str,str,str,str
"""S1-925783039""","""Orelee's Barbershop""","""1795 Westchester Drive, High P…","""US"""
"""S1-773889195""","""Prime Money""","""17560 Ellis Road, Tahlequah, O…","""US"""
"""S1-377745466""","""B+ Retail Inc""","""1712 Montebello Avenue, Phoeni…","""US"""
"""S1-133037285""","""Christ Chapel""","""2100 Cameron Drive, Unit APART…","""US"""
"""S1-755362802""","""Prabhav Business Center""","""797, Lake Town Block A, Kolkat…","""India"""



train_source2.tsv


...

entity_id,business_name,business_address,country
str,str,str,str
"""S1-714132312""","""Zephay Labs Inc""","""2621 Cotten Road, Tyler, TX""","""US"""
"""S1-106407869""","""Vision Partners Corp""","""IA, Iowa City, 1064 Newton Rd,…","""US"""
"""S1-156285671""","""<< Team Ecole""","""175 Boulevard du Président Fra…","""France"""
"""S1-689823050""","""Red Perfect Trading""","""Mirzapur, Ews 12, Uttar Prades…","""India"""
"""S1-921369899""","""ZNB Club SARL""","""Nouvelle-Aquitaine, La Teste-d…","""France"""



test_source2.tsv


entity_id,business_name,business_address,country
str,str,str,str
"""S2-192345572""","""Brahma Infosoft""","""COIMATORE COLONY, HUNSUR TQMYS…","""India"""
"""S2-566025912""","""Marina Ecole France Sarl""","""63 R. DE DIEPPE, LILLE, Hauts-…","""France"""
"""S2-158121477""","""SCI Ptit Àmicale""","""18 RUE JEN ZAY, Dunkerque, Nor…","""France"""
"""S2-89663826""","""Apex Summit""","""67 KENTUCKY ST, SALYERSVILLE, …","""US"""
"""S2-884102769""","""Fresh Truist""","""8264 FILLY COURT, ROANOKE COUN…","""US"""



test_source3.tsv


entity_id,business_name,business_address,country
str,str,str,str
"""S3-462677478""","""मॉडर्न फाइनेंस""","""No 10 Enkay Square, 448A, Udyo…","""India"""
"""S3-374810425""","""Shri Sai Infratech Co""","""3/115, East Delhi, DL""","""India"""
"""S3-198586129""","""Fractales Amis Groupe S.A.S""","""23 Rue Icmre, La Teste-de-buch…","""France"""
"""S3-10300249""","""Shri Supreme Consulting Privat…","""H.no 910 A 3503, Mumbai, महारा…","""India"""
"""S3-604980231""","""Prime Realty Ventures Public L…","""G.t. Karnal Road, Industrial A…","""India"""


## 3. Challenge schema inspection

### Inspect columns before building the model

The exact `train_ground_truth.tsv` list serialization is not visible from the uploaded notebook alone, so this cell inspects the real local file before parsing it.

The source tables must contain:

```text
entity_id
business_name
business_address
country
```


In [6]:
# ============================================================
# Cell 4 — Schema inspection
# ============================================================

REQUIRED_SOURCE_COLUMNS = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

def get_columns(path):
    return pl.read_csv(
        path,
        separator="\t",
        has_header=True,
        infer_schema=False,
        n_rows=0,
    ).columns

for source_name, path in [
    ("train_source1", train_source1_file),
    ("train_source2", train_source2_file),
    ("train_source3", train_source3_file),
    ("test_source1", test_source1_file),
    ("test_source2", test_source2_file),
    ("test_source3", test_source3_file),
]:
    cols = get_columns(path)
    missing = [c for c in REQUIRED_SOURCE_COLUMNS if c not in cols]
    print(f"{source_name}:")
    print("  columns:", cols)
    print("  missing required:", missing)

gt_columns = get_columns(ground_truth_file)
print("\ntrain_ground_truth columns:", gt_columns)


train_source1:
  columns: ['entity_id', 'business_name', 'business_address', 'country']
  missing required: []
train_source2:
  columns: ['entity_id', 'business_name', 'business_address', 'country']
  missing required: []
train_source3:
  columns: ['entity_id', 'business_name', 'business_address', 'country']
  missing required: []
test_source1:
  columns: ['entity_id', 'business_name', 'business_address', 'country']
  missing required: []
test_source2:
  columns: ['entity_id', 'business_name', 'business_address', 'country']
  missing required: []
test_source3:
  columns: ['entity_id', 'business_name', 'business_address', 'country']
  missing required: []

train_ground_truth columns: ['source1_entity_id', 'matched_entity_ids']


## 4. Ground-truth parsing

### Flexible parser

The notebook will try common ground-truth layouts:

- one Source-1 ID column plus list columns for Source 2 / Source 3;
- one row per matched pair;
- comma/semicolon/pipe separated lists.

The parsed internal representation will always be:

```text
source1_entity_id | target_source | target_entity_id
```

This makes the rest of the pipeline independent of the exact list formatting.

In [7]:
# ============================================================
# Cell 5 — Ground-truth parsing (UPDATED)
# ============================================================

import ast
import polars as pl

def safe_text(x):
    if x is None:
        return ""
    s = str(x)
    if s.lower() in {"nan", "none", "null"}:
        return ""
    return s

def parse_id_list(value):
    s = safe_text(value).strip()
    if not s or s in {"[]", "{}", "nan", "None", "null"}:
        return []

    # Python/JSON-like list.
    if s.startswith("[") or s.startswith("("):
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, (list, tuple, set)):
                return list(dict.fromkeys(str(v).strip() for v in obj if str(v).strip()))
        except Exception:
            pass

    s = s.replace(";", ",").replace("|", ",")
    return list(dict.fromkeys(x.strip() for x in s.split(",") if x.strip()))

gt_raw = pl.read_csv(
    ground_truth_file,
    separator="\t",
    has_header=True,
    infer_schema=False,
    null_values=["", "NULL", "None", "null"],
)

print("Ground-truth columns:", gt_raw.columns)

gt_rows = []

# Manually process the single 'matched_entity_ids' column
for row in gt_raw.iter_rows(named=True):
    s1 = safe_text(row.get("source1_entity_id", ""))
    if not s1:
        continue

    # Get all matches from the combined column
    matches = parse_id_list(row.get("matched_entity_ids", ""))
    
    # Route them to source2 or source3 based on their prefix
    for tid in matches:
        if tid.startswith("S2-"):
            gt_rows.append((s1, "source2", tid))
        elif tid.startswith("S3-"):
            gt_rows.append((s1, "source3", tid))

ground_truth_long = (
    pl.DataFrame(
        gt_rows,
        schema=[
            ("source1_entity_id", pl.String),
            ("target_source", pl.String),
            ("target_entity_id", pl.String),
        ],
    )
    .unique()
)

display(ground_truth_long.head(20))
print("Ground-truth positive pairs:", len(ground_truth_long))

Ground-truth columns: ['source1_entity_id', 'matched_entity_ids']


C:\Users\91894\miniconda3\envs\cuda_env\lib\functools.py:889: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  return dispatch(args[0].__class__)(*args, **kw)


source1_entity_id,target_source,target_entity_id
str,str,str
"""S1-953822129""","""source2""","""S2-372729866"""
"""S1-559924347""","""source3""","""S3-118153050"""
"""S1-220184105""","""source3""","""S3-678655007"""
"""S1-89067415""","""source3""","""S3-581751923"""
"""S1-546218228""","""source2""","""S2-493695465"""
…,…,…
"""S1-148798833""","""source2""","""S2-232864928"""
"""S1-973851703""","""source3""","""S3-479478615"""
"""S1-153344133""","""source2""","""S2-493841763"""


Ground-truth positive pairs: 7638365


## 5. 65 / 20 / 15 entity-level split

### Why the split is done by Source-1 entity

All candidate pairs belonging to one Source-1 entity must remain in the same split.

Otherwise the same Source-1 entity could leak into train and validation/test through different candidate rows.

The split is:

```text
65% train
20% validation
15% held-out test
```

The 20% validation set tunes the precision shield.

The 15% set is not used for tuning.

In [8]:
# ============================================================
# Cell 6 — 65 / 20 / 15 split
# ============================================================

s1_ids = (
    pl.scan_csv(
        train_source1_file,
        separator="\t",
        has_header=True,
        infer_schema=False,
    )
    .select("entity_id")
    .collect()["entity_id"]
    .to_list()
)

rng = np.random.default_rng(RANDOM_SEED)
perm = rng.permutation(len(s1_ids))

n = len(s1_ids)
n_train = int(n * TRAIN_FRAC)
n_val = int(n * VAL_FRAC)

split_labels = np.empty(n, dtype=object)
split_labels[perm[:n_train]] = "train65"
split_labels[perm[n_train:n_train+n_val]] = "val20"
split_labels[perm[n_train+n_val:]] = "test15"

source1_split = pl.DataFrame({
    "entity_id": s1_ids,
    "split": split_labels.tolist(),
})

split_file = os.path.join(artifact_dir, "source1_split.parquet")
os.makedirs(artifact_dir, exist_ok=True)
source1_split.write_parquet(split_file, compression="zstd")

print(source1_split.group_by("split").len())


shape: (3, 2)
┌─────────┬─────────┐
│ split   ┆ len     │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ val20   ┆ 441364  │
│ train65 ┆ 1434433 │
│ test15  ┆ 331024  │
└─────────┴─────────┘


## 6. Multilingual normalization

### Representation policy

We never replace the original data.

For each record we keep:

```text
original Unicode
normalized Unicode
transliterated representation
```

This is important for:

- Hindi
- Bengali
- Odia
- Kannada
- Gujarati
- Tamil
- Telugu
- Malayalam
- English
- French
- transliterated text
- Hinglish / mixed-language text

Hinglish written in Latin script stays as text and is later handled by the multilingual encoder.

### Missing vs contradictory fields

A missing PIN is not automatically a mismatch.

A known-but-different PIN is a separate contradiction feature.

In [9]:
# ============================================================
# Cell 7 — Normalization functions
# ============================================================

BUSINESS_STOPWORDS = {
    "private", "limited", "pvt", "ltd", "inc", "incorporated",
    "company", "co", "corporation", "corp", "llc", "llp", "plc",
    "the", "and", "of", "for", "group", "holdings", "international",
}

ADDRESS_STOPWORDS = {
    "road", "rd", "street", "st", "lane", "ln", "avenue", "ave",
    "block", "sector", "district", "near", "opposite", "opp",
    "building", "floor", "flat", "plot", "house", "no",
}

TOKEN_RE = re.compile(r"[\w]+", re.UNICODE)
PIN_RE = re.compile(r"(?<!\d)(\d{5,6}(?:-\d{4})?)(?!\d)")

ABBREVIATIONS = {
    "pvt": "private",
    "pvt.": "private",
    "ltd": "limited",
    "ltd.": "limited",
    "inc.": "incorporated",
    "co.": "company",
    "corp.": "corporation",
    "rd.": "road",
    "st.": "street",
    "ln.": "lane",
    "ave.": "avenue",
    "apt.": "apartment",
    "blvd.": "boulevard",
    "&": "and",
}

def normalize_unicode(text):
    text = safe_text(text)
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text).casefold()

    for old, new in ABBREVIATIONS.items():
        text = text.replace(old, f" {new} ")

    # Keep all Unicode letters/digits.
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()

def transliterate_text(text):
    base = normalize_unicode(text)
    if not base:
        return ""
    if unidecode is None:
        return base
    return normalize_unicode(unidecode(base))

def extract_pin(text):
    matches = PIN_RE.findall(safe_text(text))
    return matches[-1] if matches else ""

def token_list(text, stopwords=None):
    stopwords = stopwords or set()
    out = []
    for tok in TOKEN_RE.findall(safe_text(text)):
        if tok in stopwords or len(tok) < 2:
            continue
        out.append(tok)
    return list(dict.fromkeys(out))

def char_ngrams(text, n=3):
    text = re.sub(r"\s+", " ", safe_text(text))
    if not text:
        return set()
    if len(text) <= n:
        return {text}
    return {text[i:i+n] for i in range(len(text)-n+1)}

def jaccard_similarity(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def non_latin_fraction(text):
    chars = [c for c in safe_text(text) if c.isalpha()]
    if not chars:
        return 0.0
    latin = sum("LATIN" in unicodedata.name(c, "") for c in chars)
    return 1.0 - latin / len(chars)

def business_encoder_text(name, address, country):
    return f"{safe_text(name)} || {safe_text(address)} || {safe_text(country)}"


## 7. Streaming normalization: TSV → Parquet

### Why Parquet

We perform the expensive string normalization once and persist the results.

The full 1–2 GB TSV does not need to be held in one giant Python DataFrame.

In [10]:
# ============================================================
# Cell 8 — Streaming normalization
# ============================================================

def normalize_tsv_to_parquet(tsv_path, parquet_path, batch_rows=50_000):
    if os.path.exists(parquet_path):
        print("Using cached:", parquet_path)
        return

    con = duckdb.connect()
    reader = con.execute(
        """
        SELECT
            CAST(entity_id AS VARCHAR) AS entity_id,
            CAST(business_name AS VARCHAR) AS business_name,
            CAST(business_address AS VARCHAR) AS business_address,
            CAST(country AS VARCHAR) AS country
        FROM read_csv_auto(?, delim='\t', header=true, all_varchar=true)
        """,
        [tsv_path],
    ).fetch_record_batch(batch_rows)

    writer = None
    row_offset = 0
    total = 0

    try:
        for rb in reader:
            df = pl.from_arrow(rb)

            df = df.with_columns(
                pl.Series(
                    "row_idx",
                    np.arange(row_offset, row_offset + len(df), dtype=np.int64)
                )
            )

            df = df.with_columns([
                pl.col("business_name")
                    .map_elements(normalize_unicode, return_dtype=pl.String)
                    .alias("name_norm"),

                pl.col("business_address")
                    .map_elements(normalize_unicode, return_dtype=pl.String)
                    .alias("address_norm"),

                pl.col("country")
                    .map_elements(normalize_unicode, return_dtype=pl.String)
                    .alias("country_norm"),

                pl.col("business_name")
                    .map_elements(transliterate_text, return_dtype=pl.String)
                    .alias("name_trans"),

                pl.col("business_address")
                    .map_elements(transliterate_text, return_dtype=pl.String)
                    .alias("address_trans"),

                pl.col("country")
                    .map_elements(transliterate_text, return_dtype=pl.String)
                    .alias("country_trans"),

                pl.col("business_address")
                    .map_elements(extract_pin, return_dtype=pl.String)
                    .alias("pin"),
            ])

            table = df.to_arrow()

            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_path,
                    table.schema,
                    compression="zstd",
                )

            writer.write_table(table)
            row_offset += len(df)
            total += len(df)

            if total % 500_000 < batch_rows:
                print("processed rows:", total)

    finally:
        if writer is not None:
            writer.close()
        con.close()

    print("Written:", parquet_path, "| rows:", total)

os.makedirs(artifact_dir, exist_ok=True)

normalized_paths = {
    "train_source1": os.path.join(artifact_dir, "train_source1.parquet"),
    "train_source2": os.path.join(artifact_dir, "train_source2.parquet"),
    "train_source3": os.path.join(artifact_dir, "train_source3.parquet"),
    "test_source1": os.path.join(artifact_dir, "test_source1.parquet"),
    "test_source2": os.path.join(artifact_dir, "test_source2.parquet"),
    "test_source3": os.path.join(artifact_dir, "test_source3.parquet"),
}

source_tsv_map = {
    "train_source1": train_source1_file,
    "train_source2": train_source2_file,
    "train_source3": train_source3_file,
    "test_source1": test_source1_file,
    "test_source2": test_source2_file,
    "test_source3": test_source3_file,
}

for key, tsv in source_tsv_map.items():
    normalize_tsv_to_parquet(tsv, normalized_paths[key])

display(pl.read_parquet(normalized_paths["train_source1"]).head(5))


Using cached: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\train_source1.parquet
Using cached: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\train_source2.parquet
Using cached: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\train_source3.parquet
Using cached: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\test_source1.parquet
Using cached: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\test_source2.parquet
Using cached: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\test_source3.parquet


entity_id,business_name,business_address,country,row_idx,name_norm,address_norm,country_norm,name_trans,address_trans,country_trans,pin
str,str,str,str,i64,str,str,str,str,str,str,str
"""S1-925783039""","""Orelee's Barbershop""","""1795 Westchester Drive, High P…","""US""",0,"""orelee s barbershop""","""1795 westchester drive high po…","""us""","""orelee s barbershop""","""1795 westchester drive high po…","""us""",""""""
"""S1-773889195""","""Prime Money""","""17560 Ellis Road, Tahlequah, O…","""US""",1,"""prime money""","""17560 ellis road tahlequah ok""","""us""","""prime money""","""17560 ellis road tahlequah ok""","""us""","""17560"""
"""S1-377745466""","""B+ Retail Inc""","""1712 Montebello Avenue, Phoeni…","""US""",2,"""b retail inc""","""1712 montebello avenue phoenix…","""us""","""b retail inc""","""1712 montebello avenue phoenix…","""us""",""""""
"""S1-133037285""","""Christ Chapel""","""2100 Cameron Drive, Unit APART…","""US""",3,"""christ chapel""","""2100 cameron drive unit apartm…","""us""","""christ chapel""","""2100 cameron drive unit apartm…","""us""",""""""
"""S1-755362802""","""Prabhav Business Center""","""797, Lake Town Block A, Kolkat…","""India""",4,"""prabhav business center""","""797 lake town block a kolkata …","""india""","""prabhav business center""","""797 lake town block a kolkata …","""india""",""""""


## 8. Lexical blocking

### Blocking objective

Blocking is **recall-focused**.

The final classifier can reject false candidates, but it cannot recover a true match that was never included in the candidate set.

We therefore use multiple independent blocking passes and take their union:

1. exact normalized name + country
2. exact normalized name globally
3. transliterated name prefix/suffix + country
4. informative business-name tokens + country
5. informative address tokens + country
6. address prefix + country
7. PIN
8. multilingual ANN retrieval

### Generic block protection

Very large blocks are excluded so a generic token such as `company` does not explode into millions of candidates.

In [11]:
# ============================================================
# Cell 9 — Lexical blocking index
# ============================================================

def make_block_keys(row):
    country = row["country_trans"] or row["country_norm"] or "_"
    name_norm = row["name_norm"]
    name_trans = row["name_trans"]
    address_trans = row["address_trans"]
    pin = row["pin"]

    keys = set()

    if name_norm:
        keys.add(f"NFULL|{country}|{name_norm}")
        keys.add(f"NFULL|*|{name_norm}")

    if len(name_trans) >= 5:
        keys.add(f"NPFX5|{country}|{name_trans[:5]}")
        keys.add(f"NSFX5|{country}|{name_trans[-5:]}")

    name_tokens = sorted(
        set(token_list(name_trans, BUSINESS_STOPWORDS)),
        key=lambda x: (-len(x), x)
    )

    for tok in name_tokens[:3]:
        keys.add(f"NTOK|{country}|{tok}")

    if pin:
        keys.add(f"PIN|{pin}")

    if address_trans:
        keys.add(f"APFX5|{country}|{address_trans[:5]}")

        address_tokens = sorted(
            set(token_list(address_trans, ADDRESS_STOPWORDS)),
            key=lambda x: (-len(x), x)
        )

        for tok in address_tokens[:3]:
            keys.add(f"ATOK|{country}|{tok}")

    return list(keys)[:MAX_BLOCK_KEYS_PER_ENTITY]


def build_block_index(parquet_path, db_path):
    if os.path.exists(db_path):
        print("Using cached block index:", db_path)
        return

    con = sqlite3.connect(db_path)
    cur = con.cursor()

    cur.execute("PRAGMA journal_mode=WAL")
    cur.execute("PRAGMA synchronous=OFF")

    cur.execute("""
        CREATE TABLE records (
            row_idx INTEGER PRIMARY KEY,
            entity_id TEXT NOT NULL
        )
    """)

    cur.execute("""
        CREATE TABLE blocks (
            block_key TEXT NOT NULL,
            row_idx INTEGER NOT NULL
        )
    """)

    con.commit()

    duck = duckdb.connect()
    reader = duck.execute(
        """
        SELECT row_idx, entity_id, name_norm, name_trans,
               address_norm, address_trans,
               country_norm, country_trans, pin
        FROM read_parquet(?)
        ORDER BY row_idx
        """,
        [parquet_path],
    ).fetch_record_batch(50_000)

    try:
        for rb in reader:
            df = pl.from_arrow(rb)

            cur.executemany(
                "INSERT INTO records(row_idx, entity_id) VALUES (?, ?)",
                [
                    (int(r["row_idx"]), safe_text(r["entity_id"]))
                    for r in df.iter_rows(named=True)
                ],
            )

            block_rows = []

            for r in df.iter_rows(named=True):
                for key in make_block_keys(r):
                    block_rows.append((key, int(r["row_idx"])))

            cur.executemany(
                "INSERT INTO blocks(block_key, row_idx) VALUES (?, ?)",
                block_rows
            )

            con.commit()
    finally:
        duck.close()

    cur.execute("CREATE INDEX idx_blocks_key ON blocks(block_key)")
    cur.execute("""
        CREATE TABLE block_stats AS
        SELECT block_key, COUNT(*) AS cnt
        FROM blocks
        GROUP BY block_key
    """)
    cur.execute("CREATE INDEX idx_block_stats_key ON block_stats(block_key)")
    con.commit()
    con.close()

    print("Created:", db_path)

train_block_db = {}
test_block_db = {}

for source_num in [2, 3]:
    train_db = os.path.join(artifact_dir, f"train_source{source_num}_blocks.sqlite")
    test_db = os.path.join(artifact_dir, f"test_source{source_num}_blocks.sqlite")

    build_block_index(
        normalized_paths[f"train_source{source_num}"],
        train_db,
    )

    build_block_index(
        normalized_paths[f"test_source{source_num}"],
        test_db,
    )

    train_block_db[source_num] = train_db
    test_block_db[source_num] = test_db


Using cached block index: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\train_source2_blocks.sqlite
Using cached block index: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\test_source2_blocks.sqlite
Using cached block index: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\train_source3_blocks.sqlite
Using cached block index: C:\Users\91894\Downloads\6ab10eb3b23ba_student_resource\student_resource\artifacts\test_source3_blocks.sqlite


## 9. Local multilingual embedding model

### GPU policy

The embedding model is optional but recommended for the strongest multilingual system.

The entire dataset is **never** transferred to the GPU.

Only small batches are encoded.

For your 4 GB RTX 3050:

```text
start batch size = 8
if OOM → 4
if OOM → 2
if OOM → 1
```

The embeddings are saved to disk as `float16`.

### No external augmentation

This notebook makes no web/API/geocoding calls.

The multilingual model must already exist locally and be permitted by the challenge rules.

In [12]:
# ============================================================
# Cell 10 — Load local multilingual encoder
# ============================================================

USE_EMBEDDINGS = (
    torch is not None
    and SentenceTransformer is not None
    and faiss is not None
    and os.path.isdir(embedding_model_path)
)

if USE_EMBEDDINGS:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Loading local model:", embedding_model_path)
    encoder = SentenceTransformer(embedding_model_path, device=device)
    encoder.eval()

    if device == "cuda":
        try:
            encoder.half()
        except Exception:
            pass

    print("Embedding device:", device)
else:
    encoder = None
    device = "cpu"
    print(
        "Semantic embedding stage disabled.\n"
        "Provide a local permitted model at embedding_model_path and ensure "
        "sentence-transformers + faiss are installed."
    )


Semantic embedding stage disabled.
Provide a local permitted model at embedding_model_path and ensure sentence-transformers + faiss are installed.


## 10. Batched GPU embedding

### VRAM-safe encoder

This function never sends all records to the GPU.

It adaptively decreases the batch size when CUDA reports an out-of-memory condition.

In [13]:
# ============================================================
# Cell 11 — VRAM-safe embedding helpers
# ============================================================

def encode_batch_adaptive(texts, batch_size=GPU_EMBED_BATCH_SIZE):
    if encoder is None:
        raise RuntimeError("Embedding encoder is not enabled.")

    current_bs = batch_size

    while current_bs >= 1:
        try:
            if torch is not None and device == "cuda":
                with torch.inference_mode():
                    with torch.autocast("cuda", dtype=torch.float16):
                        emb = encoder.encode(
                            texts,
                            batch_size=current_bs,
                            show_progress_bar=False,
                            convert_to_numpy=True,
                            normalize_embeddings=True,
                        )
            else:
                emb = encoder.encode(
                    texts,
                    batch_size=current_bs,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                )

            return np.asarray(emb, dtype=np.float32)

        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise

            print(f"CUDA OOM at batch={current_bs}; retrying with {current_bs // 2}")
            if torch is not None and torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

            current_bs //= 2

    raise RuntimeError("Embedding failed even with batch size 1.")


def build_embeddings_memmap(parquet_path, output_path, batch_rows=256):
    if os.path.exists(output_path):
        arr = np.load(output_path, mmap_mode="r")
        print("Using cached embeddings:", output_path, arr.shape)
        return output_path, arr.shape[1]

    if encoder is None:
        return None, None

    con = duckdb.connect()
    n_rows = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [parquet_path]
    ).fetchone()[0]
    con.close()

    # Determine embedding dimension from one small batch.
    con = duckdb.connect()
    first = con.execute(
        """
        SELECT row_idx, business_name, business_address, country
        FROM read_parquet(?)
        ORDER BY row_idx
        LIMIT 8
        """,
        [parquet_path],
    ).fetchall()
    con.close()

    first_texts = [
        business_encoder_text(r[1], r[2], r[3])
        for r in first
    ]

    first_emb = encode_batch_adaptive(first_texts, 4)
    dim = first_emb.shape[1]

    mmap = np.lib.format.open_memmap(
        output_path,
        mode="w+",
        dtype=np.float16,
        shape=(int(n_rows), int(dim)),
    )

    # First rows.
    first_idx = np.array([r[0] for r in first], dtype=np.int64)
    mmap[first_idx] = first_emb.astype(np.float16)

    con = duckdb.connect()
    reader = con.execute(
        """
        SELECT row_idx, business_name, business_address, country
        FROM read_parquet(?)
        WHERE row_idx >= ?
        ORDER BY row_idx
        """,
        [parquet_path, int(first_idx.max()) + 1],
    ).fetch_record_batch(batch_rows)

    total_done = len(first)

    try:
        for rb in reader:
            df = pl.from_arrow(rb)

            idxs = df["row_idx"].to_numpy()
            texts = [
                business_encoder_text(a, b, c)
                for a, b, c in zip(
                    df["business_name"].to_list(),
                    df["business_address"].to_list(),
                    df["country"].to_list(),
                )
            ]

            emb = encode_batch_adaptive(texts, GPU_EMBED_BATCH_SIZE)
            mmap[idxs] = emb.astype(np.float16)

            total_done += len(idxs)

            if total_done % 50_000 < batch_rows:
                print(f"embedded {total_done:,}/{n_rows:,}")

    finally:
        con.close()

    mmap.flush()
    del mmap

    print("Saved:", output_path)
    return output_path, dim


## 11. Build Source-2 / Source-3 semantic indexes

### Index placement

The embedding vectors are stored on disk.

FAISS is intentionally CPU-side so the 4 GB GPU remains available for the encoder.

For the challenge test set, we will later retrieve approximately `ANN_K` semantic candidates per Source-1 record.

In [14]:
# ============================================================
# Cell 12 — Build embeddings + FAISS indexes
# ============================================================

def build_faiss_index(emb_path, index_path):
    if os.path.exists(index_path):
        print("Using cached FAISS:", index_path)
        return index_path

    emb = np.load(emb_path, mmap_mode="r")
    n, dim = emb.shape

    # For large collections, use compressed IVF-PQ.
    if n <= 1_000_000:
        index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efConstruction = 120
        index.hnsw.efSearch = max(64, ANN_K * 2)

        chunk = 20_000
        for start in range(0, n, chunk):
            end = min(n, start + chunk)
            index.add(np.asarray(emb[start:end], dtype=np.float32))
    else:
        nlist = max(1024, min(8192, int(4 * math.sqrt(n))))
        m = 32 if dim % 32 == 0 else 16

        quantizer = faiss.IndexFlatIP(dim)
        index = faiss.IndexIVFPQ(
            quantizer, dim, nlist, m, 8,
            faiss.METRIC_INNER_PRODUCT
        )

        train_n = min(n, 100_000)
        sample_idx = np.linspace(
            0, n - 1, train_n, dtype=np.int64
        )
        train_x = np.asarray(emb[sample_idx], dtype=np.float32)

        index.train(train_x)

        chunk = 20_000
        for start in range(0, n, chunk):
            end = min(n, start + chunk)
            index.add(np.asarray(emb[start:end], dtype=np.float32))

        index.nprobe = min(64, max(8, nlist // 64))

    faiss.write_index(index, index_path)
    print("Created FAISS:", index_path)
    return index_path


embedding_paths = {}
faiss_paths = {}

if USE_EMBEDDINGS:
    for split in ["train", "test"]:
        for source_num in [1, 2, 3]:
            key = f"{split}_source{source_num}"
            emb_path = os.path.join(
                artifact_dir, f"{key}_embeddings.npy"
            )

            build_embeddings_memmap(
                normalized_paths[key],
                emb_path,
            )

            embedding_paths[key] = emb_path

        for source_num in [2, 3]:
            key = f"{split}_source{source_num}"
            idx_path = os.path.join(
                artifact_dir, f"{key}_faiss.index"
            )

            build_faiss_index(
                embedding_paths[key],
                idx_path,
            )

            faiss_paths[key] = idx_path
else:
    print("Embedding/FAISS stages skipped.")


Embedding/FAISS stages skipped.


## 12. Candidate generation

### Exact candidate set used by the final model

For each Source-1 record:

```text
lexical candidates
        UNION
semantic ANN candidates
        =
final candidate set
```

No pair outside this set will be passed to the final inference model.

That is exactly what we will persist as:

```text
output/candidate_pairs.tsv
```

for the official test data.

### Candidate metadata

We keep:

- Source-1 row index
- target row index
- target source
- number of independent lexical blocks hit
- semantic retrieval score

The first four are useful audit information; the final model uses the candidate set itself.

In [15]:
# ============================================================
# Cell 13 — High-Performance Vectorized Candidate Generation
# ============================================================

def batch_lexical_candidates(block_db_path, keys_list, max_block_size=MAX_BLOCK_SIZE):
    """
    Batched block lookup: fetches candidate hits for an entire batch of records
    in a single SQL query using a temporary table join instead of querying SQLite row-by-row.
    """
    if not keys_list:
        return []

    con = sqlite3.connect(block_db_path)
    cur = con.cursor()
    
    # Flatten all keys for batch filtering
    all_keys = list({k for keys in keys_list for k in keys})
    if not all_keys:
        con.close()
        return [{} for _ in keys_list]

    # Query valid blocks under size limit
    cur.execute("""
        CREATE TEMP TABLE query_keys (block_key TEXT);
    """)
    cur.executemany("INSERT INTO query_keys VALUES (?)", [(k,) for k in all_keys])
    
    valid_blocks = set(r[0] for r in cur.execute("""
        SELECT b.block_key 
        FROM query_keys q
        JOIN block_stats b ON q.block_key = b.block_key
        WHERE b.cnt <= ?
    """, (max_block_size,)).fetchall())

    # BULK BATCH QUERY OPTIMIZATION
    cur.execute("CREATE TEMP TABLE batch_keys (rec_id INT, block_key TEXT)")
    
    insert_rows = []
    for i, keys in enumerate(keys_list):
        active = [k for k in keys if k in valid_blocks][:MAX_BLOCK_KEYS_PER_ENTITY]
        for k in active:
            insert_rows.append((i, k))
            
    if not insert_rows:
        cur.execute("DROP TABLE query_keys")
        cur.execute("DROP TABLE batch_keys")
        con.close()
        return [{} for _ in keys_list]

    cur.executemany("INSERT INTO batch_keys VALUES (?, ?)", insert_rows)
    
    batch_hits = cur.execute("""
        SELECT bk.rec_id, b.row_idx, COUNT(DISTINCT b.block_key)
        FROM batch_keys bk
        JOIN blocks b ON bk.block_key = b.block_key
        GROUP BY bk.rec_id, b.row_idx
    """).fetchall()
    
    cur.execute("DROP TABLE batch_keys")
    cur.execute("DROP TABLE query_keys")
    con.close()
    
    results = [{} for _ in keys_list]
    for rec_id, tidx, hits in batch_hits:
        results[rec_id][int(tidx)] = int(hits)
        
    return results


def generate_candidate_parquet(
    s1_parquet,
    target_parquet,
    target_block_db,
    target_embedding_path=None,
    target_faiss_path=None,
    s1_embedding_path=None,
    output_parquet=None,
    s1_filter=None,
):
    if output_parquet and os.path.exists(output_parquet):
        print("Using cached candidates:", output_parquet)
        return output_parquet

    target_index = (
        faiss.read_index(target_faiss_path)
        if target_faiss_path and os.path.exists(target_faiss_path)
        else None
    )

    s1_emb = (
        np.load(s1_embedding_path, mmap_mode="r")
        if s1_embedding_path and os.path.exists(s1_embedding_path)
        else None
    )

    con = duckdb.connect()
    where_sql = ""
    params = [s1_parquet]

    if s1_filter:
        where_sql = """
        INNER JOIN read_parquet(?) sp ON s.entity_id = sp.entity_id
        WHERE sp.split = ?
        """
        params = [s1_parquet, s1_filter[0], s1_filter[1]]

    query = f"""
        SELECT s.row_idx, s.entity_id, s.name_norm, s.name_trans,
               s.address_norm, s.address_trans, s.country_norm, s.country_trans, s.pin
        FROM read_parquet(?) s
        {where_sql}
        ORDER BY s.row_idx
    """

    reader = con.execute(query, params).fetch_record_batch(1024)

    schema = pa.schema([
        ("s1_row_idx", pa.int64()),
        ("target_row_idx", pa.int64()),
        ("block_hits", pa.int32()),
        ("retrieval_score", pa.float32()),
    ])

    writer = pq.ParquetWriter(output_parquet, schema, compression="zstd")
    total = 0

    try:
        for rb in reader:
            df = pl.from_arrow(rb)
            ann_scores = ann_ids = None

            if target_index is not None and s1_emb is not None:
                query_vecs = np.asarray(s1_emb[df["row_idx"].to_numpy()], dtype=np.float32)
                ann_scores, ann_ids = target_index.search(query_vecs, ANN_K)

            batch_keys = [make_block_keys(row) for row in df.iter_rows(named=True)]
            batch_lex_hits = batch_lexical_candidates(target_block_db, batch_keys)

            batch_rows = []
            for i, row in enumerate(df.iter_rows(named=True)):
                lex = batch_lex_hits[i]
                merged = {idx: (hits, -1.0) for idx, hits in lex.items()}

                if ann_ids is not None:
                    for j in range(len(ann_ids[i])):
                        tidx = int(ann_ids[i][j])
                        if tidx < 0:
                            continue
                        score = float(ann_scores[i][j])
                        old_hits, old_score = merged.get(tidx, (0, -1.0))
                        merged[tidx] = (old_hits, max(old_score, score))

                for tidx, (hits, score) in merged.items():
                    batch_rows.append({
                        "s1_row_idx": int(row["row_idx"]),
                        "target_row_idx": int(tidx),
                        "block_hits": int(hits),
                        "retrieval_score": float(score),
                    })

            if batch_rows:
                writer.write_table(pa.Table.from_pylist(batch_rows, schema=schema))
                total += len(batch_rows)

    finally:
        writer.close()
        con.close()

    print("Candidate rows generated:", total)
    return output_parquet

## 13. Build train/validation/test candidate sets

### Candidate recall ceiling

We measure:

\[
CandidateRecall =
\frac{\text{true matches present in candidates}}
{\text{all true matches}}
\]

before model training.

This is one of the most important diagnostics in the challenge.

In [ ]:
# ============================================================
# Cell 14 — Build candidate sets for 65/20/15
# ============================================================

candidate_paths = {}

for split_name in ["train65", "val20", "test15"]:
    for source_num in [2, 3]:

        out_path = os.path.join(
            artifact_dir,
            f"candidates_{split_name}_source{source_num}.parquet"
        )

        candidate_paths[(split_name, source_num)] = generate_candidate_parquet(
            s1_parquet=normalized_paths["train_source1"],
            target_parquet=normalized_paths[f"train_source{source_num}"],
            target_block_db=train_block_db[source_num],
            target_embedding_path=embedding_paths.get(f"train_source{source_num}"),
            target_faiss_path=faiss_paths.get(f"train_source{source_num}"),
            s1_embedding_path=embedding_paths.get("train_source1"),
            output_parquet=out_path,
            s1_filter=(split_file, split_name),
        )

print("Candidate files prepared successfully.")

C:\Users\91894\AppData\Local\Temp\ipykernel_32532\287219168.py:115: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  reader = con.execute(query, params).fetch_record_batch(1024)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Candidate recall evaluator

A blocker with 80% candidate recall cannot possibly produce more than roughly 80% recall downstream on those missed pairs.

So this cell should be inspected before serious model tuning.

In [ ]:
# ============================================================
# Cell 15 — Candidate recall evaluation
# ============================================================

def candidate_recall(
    candidate_path,
    s1_parquet,
    target_parquet,
    gt_long,
    target_source,
    split_file,
    split_name,
):
    gt = gt_long.filter(pl.col("target_source") == target_source)

    split_ids = (
        pl.read_parquet(split_file)
        .filter(pl.col("split") == split_name)
        .select("entity_id")
    )

    gt = gt.join(
        split_ids,
        left_on="source1_entity_id",
        right_on="entity_id",
        how="inner"
    ).drop("entity_id")

    if len(gt) == 0:
        return 1.0

    con = duckdb.connect()

    hit_count = con.execute(
        """
        WITH cand AS (
            SELECT s.entity_id AS s1_id, t.entity_id AS t_id
            FROM read_parquet(?) c
            JOIN read_parquet(?) s ON s.row_idx = c.s1_row_idx
            JOIN read_parquet(?) t ON t.row_idx = c.target_row_idx
        ),
        truth AS (
            SELECT source1_entity_id AS s1_id, target_entity_id AS t_id
            FROM read_parquet(?)
        )
        SELECT COUNT(*)
        FROM truth g
        JOIN cand c ON g.s1_id = c.s1_id AND g.t_id = c.t_id
        """,
        [
            candidate_path,
            s1_parquet,
            target_parquet,
            os.path.join(artifact_dir, "gt_temp.parquet"),
        ],
    ).fetchone()[0]

    con.close()
    total = len(gt)
    return hit_count / total if total else 1.0

gt_temp = os.path.join(artifact_dir, "gt_temp.parquet")
ground_truth_long.write_parquet(gt_temp, compression="zstd")

for split_name in ["train65", "val20", "test15"]:
    for source_num in [2, 3]:
        rec = candidate_recall(
            candidate_paths[(split_name, source_num)],
            normalized_paths["train_source1"],
            normalized_paths[f"train_source{source_num}"],
            ground_truth_long,
            f"source{source_num}",
            split_file,
            split_name,
        )
        print(f"{split_name} -> Source {source_num} candidate recall = {rec:.5f}")

## 14. Pairwise feature engineering

### Features

Each candidate pair receives:

**Name**

- normalized exact equality
- Levenshtein normalized similarity
- Jaro-like ratio
- token-set ratio
- token Jaccard
- character 3-gram Jaccard
- TF-IDF cosine
- length ratio

**Address**

- Levenshtein normalized similarity
- Jaro-like ratio
- token-set ratio
- token Jaccard
- character 3-gram Jaccard
- TF-IDF cosine
- length ratio

**Structured**

- country equality
- country transliteration equality
- country missing
- PIN equality
- PIN missing
- PIN conflict

**Semantic**

- multilingual embedding cosine

**Blocking**

- number of independent lexical block hits
- ANN retrieval score

In [ ]:
# ============================================================
# Cell 16 — Vectorized Feature Utilities
# ============================================================
from rapidfuzz import process, fuzz, distance

FEATURE_NAMES = [
    "name_exact",
    "name_levenshtein",
    "name_ratio",
    "name_token_set_ratio",
    "name_length_ratio",

    "address_levenshtein",
    "address_ratio",
    "address_token_set_ratio",
    "address_length_ratio",

    "country_match",
    "country_trans_match",
    "country_missing",

    "pin_match",
    "pin_missing",
    "pin_conflict",

    "embedding_cosine",
    "block_hits",
    "retrieval_score",
    "ann_present",
]

print("Feature definitions compiled. Total features:", len(FEATURE_NAMES))

## 15. Build labeled training pairs

### Hard-negative mining

For each Source-1 entity we retain:

- all available positive candidates;
- the strongest negative candidates by retrieval/blocking evidence.

This is much more useful than generating only random, easy negatives.

The examples near the decision boundary are the ones that teach the model to avoid false merges.

In [ ]:
# ============================================================
# Cell 17 — Label Candidate Pairs & Hard Negative Mining
# ============================================================

def build_labeled_pairs(
    candidate_path,
    s1_parquet,
    target_parquet,
    target_source,
    split_name,
    max_pairs=MAX_TRAIN_PAIRS_PER_SOURCE,
):
    valid_ids = set(
        source1_split
        .filter(pl.col("split") == split_name)
        .get_column("entity_id")
        .to_list()
    )

    truth = defaultdict(set)
    for r in ground_truth_long.filter(pl.col("target_source") == target_source).iter_rows(named=True):
        if r["source1_entity_id"] in valid_ids:
            truth[r["source1_entity_id"]].add(r["target_entity_id"])

    con = duckdb.connect()
    df = con.execute(
        """
        SELECT c.s1_row_idx, c.target_row_idx, c.block_hits, c.retrieval_score,
               s.entity_id AS s1_id, t.entity_id AS target_id
        FROM read_parquet(?) c
        JOIN read_parquet(?) s ON s.row_idx = c.s1_row_idx
        JOIN read_parquet(?) t ON t.row_idx = c.target_row_idx
        ORDER BY c.s1_row_idx, c.retrieval_score DESC, c.block_hits DESC
        """,
        [candidate_path, s1_parquet, target_parquet],
    ).pl()
    con.close()

    positives = []
    negative_rows = defaultdict(list)

    for row in df.iter_rows(named=True):
        s1_id = row["s1_id"]
        target_id = row["target_id"]
        is_match = int(target_id in truth.get(s1_id, set()))

        item = (
            int(row["s1_row_idx"]),
            int(row["target_row_idx"]),
            int(row["block_hits"]),
            float(row["retrieval_score"]),
            is_match,
        )

        if is_match == 1:
            positives.append(item)
        else:
            negative_rows[s1_id].append(item)

    negatives = []
    for s1_id, rows in negative_rows.items():
        rows = sorted(rows, key=lambda x: (x[3], x[2]), reverse=True)
        negatives.extend(rows[:HARD_NEGATIVES_PER_ENTITY])

    all_rows = positives + negatives

    if len(all_rows) > max_pairs:
        rng = np.random.default_rng(RANDOM_SEED)
        pos_rows = [r for r in all_rows if r[-1] == 1]
        neg_rows = [r for r in all_rows if r[-1] == 0]

        remaining = max_pairs - min(len(pos_rows), max_pairs)
        if len(pos_rows) >= max_pairs:
            all_rows = pos_rows[:max_pairs]
        else:
            idx = rng.choice(len(neg_rows), size=min(remaining, len(neg_rows)), replace=False)
            all_rows = pos_rows + [neg_rows[i] for i in idx]

    return pl.DataFrame(
        all_rows,
        schema=[
            ("s1_row_idx", pl.Int64),
            ("target_row_idx", pl.Int64),
            ("block_hits", pl.Int32),
            ("retrieval_score", pl.Float32),
            ("label", pl.Int8),
        ],
    )

labeled_pair_paths = {}

for source_num in [2, 3]:
    p = os.path.join(artifact_dir, f"labeled_train_source{source_num}.parquet")
    if not os.path.exists(p):
        labeled = build_labeled_pairs(
            candidate_paths[("train65", source_num)],
            normalized_paths["train_source1"],
            normalized_paths[f"train_source{source_num}"],
            f"source{source_num}",
            "train65",
        )
        labeled.write_parquet(p, compression="zstd")
    else:
        labeled = pl.read_parquet(p)

    labeled_pair_paths[source_num] = p
    print(f"Source {source_num}: rows={len(labeled):,}, positive={(labeled['label'] == 1).sum():,}, negative={(labeled['label'] == 0).sum():,}")

## 16. Convert candidate pairs into ML features

### Memory strategy

Feature extraction is performed in chunks.

Only the current feature batch enters the Python/NumPy model matrix.

This avoids constructing a giant feature matrix for millions of candidate pairs.

In [ ]:
# ============================================================
# Cell 18 — Fully Vectorized Columnar Feature Extractor
# ============================================================

def build_features_from_pairs(
    pair_parquet,
    s1_parquet,
    target_parquet,
    s1_embedding_path=None,
    target_embedding_path=None,
    output_feature_parquet=None,
    include_label=False,
    batch_size=100_000,
):
    """
    Executes C++ RapidFuzz array operations and Polars columnar transformations
    over Arrow record batches, eliminating the slow Python iter_rows bottleneck.
    """
    if output_feature_parquet and os.path.exists(output_feature_parquet):
        print("Using cached features:", output_feature_parquet)
        return output_feature_parquet

    s1_emb = np.load(s1_embedding_path, mmap_mode="r") if (s1_embedding_path and os.path.exists(s1_embedding_path)) else None
    target_emb = np.load(target_embedding_path, mmap_mode="r") if (target_embedding_path and os.path.exists(target_embedding_path)) else None

    con = duckdb.connect()
    label_sql = ", p.label" if include_label else ""

    query = f"""
        SELECT
            p.s1_row_idx, p.target_row_idx, p.block_hits, p.retrieval_score {label_sql},
            s.name_norm AS s1_name_norm, s.name_trans AS s1_name_trans,
            s.address_norm AS s1_address_norm, s.address_trans AS s1_address_trans,
            s.country_norm AS s1_country_norm, s.country_trans AS s1_country_trans, s.pin AS s1_pin,

            t.name_norm AS t_name_norm, t.name_trans AS t_name_trans,
            t.address_norm AS t_address_norm, t.address_trans AS t_address_trans,
            t.country_norm AS t_country_norm, t.country_trans AS t_country_trans, t.pin AS t_pin
        FROM read_parquet(?) p
        JOIN read_parquet(?) s ON s.row_idx = p.s1_row_idx
        JOIN read_parquet(?) t ON t.row_idx = p.target_row_idx
        ORDER BY p.s1_row_idx, p.target_row_idx
    """

    reader = con.execute(query, [pair_parquet, s1_parquet, target_parquet]).fetch_record_batch(batch_size)

    schema_fields = [
        ("s1_row_idx", pa.int64()),
        ("target_row_idx", pa.int64()),
    ] + [(name, pa.float32()) for name in FEATURE_NAMES]

    if include_label:
        schema_fields.append(("label", pa.int8()))

    writer = pq.ParquetWriter(output_feature_parquet, pa.schema(schema_fields), compression="zstd")

    try:
        for rb in reader:
            df = pl.from_arrow(rb)

            s1_names = df["s1_name_trans"].fill_null("").to_numpy()
            t_names = df["t_name_trans"].fill_null("").to_numpy()
            s1_addrs = df["s1_address_trans"].fill_null("").to_numpy()
            t_addrs = df["t_address_trans"].fill_null("").to_numpy()

            n_samples = len(df)

            # 1. C++ Vectorized Distance Calculation
            name_lev = distance.Levenshtein.normalized_similarity(s1_names, t_names, processor=None)
            addr_lev = distance.Levenshtein.normalized_similarity(s1_addrs, t_addrs, processor=None)

            name_ratio = np.array([fuzz.ratio(a, b) / 100.0 for a, b in zip(s1_names, t_names)], dtype=np.float32)
            name_token_set = np.array([fuzz.token_set_ratio(a, b) / 100.0 for a, b in zip(s1_names, t_names)], dtype=np.float32)

            addr_ratio = np.array([fuzz.ratio(a, b) / 100.0 for a, b in zip(s1_addrs, t_addrs)], dtype=np.float32)
            addr_token_set = np.array([fuzz.token_set_ratio(a, b) / 100.0 for a, b in zip(s1_addrs, t_addrs)], dtype=np.float32)

            # 2. Batch Embedding Cosine Calculation
            if s1_emb is not None and target_emb is not None:
                s1_idx = df["s1_row_idx"].to_numpy()
                t_idx = df["target_row_idx"].to_numpy()

                v1 = np.asarray(s1_emb[s1_idx], dtype=np.float32)
                v2 = np.asarray(target_emb[t_idx], dtype=np.float32)

                norms = (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1))
                norms[norms == 0] = 1.0
                cos_sim = np.sum(v1 * v2, axis=1) / norms
            else:
                cos_sim = np.zeros(n_samples, dtype=np.float32)

            # 3. Columnar Expression Assembly
            feat_df = df.with_columns([
                (pl.col("s1_name_norm") == pl.col("t_name_norm")).cast(pl.Float32).alias("name_exact"),
                pl.Series("name_levenshtein", name_lev, dtype=pl.Float32),
                pl.Series("name_ratio", name_ratio, dtype=pl.Float32),
                pl.Series("name_token_set_ratio", name_token_set, dtype=pl.Float32),
                
                (pl.min_horizontal(pl.col("s1_name_trans").str.len_bytes(), pl.col("t_name_trans").str.len_bytes()) /
                 pl.max_horizontal(pl.col("s1_name_trans").str.len_bytes(), pl.col("t_name_trans").str.len_bytes()).clip(1, None)).cast(pl.Float32).alias("name_length_ratio"),

                pl.Series("address_levenshtein", addr_lev, dtype=pl.Float32),
                pl.Series("address_ratio", addr_ratio, dtype=pl.Float32),
                pl.Series("address_token_set_ratio", addr_token_set, dtype=pl.Float32),

                (pl.min_horizontal(pl.col("s1_address_trans").str.len_bytes(), pl.col("t_address_trans").str.len_bytes()) /
                 pl.max_horizontal(pl.col("s1_address_trans").str.len_bytes(), pl.col("t_address_trans").str.len_bytes()).clip(1, None)).cast(pl.Float32).alias("address_length_ratio"),

                (pl.col("s1_country_norm") == pl.col("t_country_norm")).cast(pl.Float32).alias("country_match"),
                (pl.col("s1_country_trans") == pl.col("t_country_trans")).cast(pl.Float32).alias("country_trans_match"),
                (pl.col("s1_country_norm").is_null() | pl.col("t_country_norm").is_null()).cast(pl.Float32).alias("country_missing"),

                ((pl.col("s1_pin") == pl.col("t_pin")) & pl.col("s1_pin").is_not_null()).cast(pl.Float32).alias("pin_match"),
                (pl.col("s1_pin").is_null() | pl.col("t_pin").is_null()).cast(pl.Float32).alias("pin_missing"),
                ((pl.col("s1_pin") != pl.col("t_pin")) & pl.col("s1_pin").is_not_null() & pl.col("t_pin").is_not_null()).cast(pl.Float32).alias("pin_conflict"),

                pl.Series("embedding_cosine", cos_sim, dtype=pl.Float32),
                pl.col("block_hits").cast(pl.Float32).alias("block_hits"),
                pl.col("retrieval_score").cast(pl.Float32).alias("retrieval_score"),
                (pl.col("retrieval_score") > -0.5).cast(pl.Float32).alias("ann_present")
            ])

            out_cols = ["s1_row_idx", "target_row_idx"] + FEATURE_NAMES
            if include_label:
                out_cols.append("label")

            writer.write_table(feat_df.select(out_cols).to_arrow())

    finally:
        writer.close()
        con.close()

    print("Features generated successfully:", output_feature_parquet)
    return output_feature_parquet

## 17. Train the pairwise ML classifier

### Model

The classifier learns:

\[
P(\text{same entity}\mid\text{name,address,country,PIN,semantic features})
\]

We use a tree-based classifier because the inputs are heterogeneous similarity features.

The GPU is not needed for this stage.

This is deliberate: your GPU is reserved for the neural embedding workload.

In [ ]:
# ============================================================
# Cell 19 — Train Source-2 and Source-3 pair models
# ============================================================

def train_pair_classifier(feature_parquet, model_path):
    features = pl.read_parquet(feature_parquet)

    X = features.select(FEATURE_NAMES).to_numpy().astype(np.float32)
    y = features["label"].to_numpy().astype(np.int8)

    positives = int((y == 1).sum())
    negatives = int((y == 0).sum())

    print(
        f"training rows={len(y):,}, "
        f"positive={positives:,}, negative={negatives:,}"
    )

    if positives == 0 or negatives == 0:
        raise ValueError("Need both positive and negative training pairs.")

    # Modest negative weighting; final threshold is optimized on validation.
    pos_weight = min(3.0, max(1.0, negatives / max(1, positives)))
    weights = np.where(y == 1, 1.0, pos_weight).astype(np.float32)

    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.10,
        random_state=RANDOM_SEED,
    )

    model.fit(X, y, sample_weight=weights)
    joblib.dump(
        {
            "model": model,
            "feature_names": FEATURE_NAMES,
        },
        model_path,
    )

    return model


train_feature_paths = {}
models = {}

for source_num in [2, 3]:
    feature_path = os.path.join(
        artifact_dir,
        f"train_features_source{source_num}.parquet"
    )

    build_features_from_pairs(
        labeled_pair_paths[source_num],
        normalized_paths["train_source1"],
        normalized_paths[f"train_source{source_num}"],
        embedding_paths.get("train_source1"),
        embedding_paths.get(f"train_source{source_num}"),
        feature_path,
        include_label=True,
    )

    train_feature_paths[source_num] = feature_path

    model_path = os.path.join(
        artifact_dir,
        f"pair_model_source{source_num}.joblib"
    )

    models[source_num] = train_pair_classifier(
        feature_path,
        model_path,
    )

    print("Saved:", model_path)


## 18. Validation scoring and the precision shield

### Decision rule

The pair classifier produces probabilities.

We do **not** automatically predict every candidate above an arbitrary similarity threshold.

We tune:

1. match probability threshold;
2. stricter requirement for semantic-only candidates;
3. ambiguity margin.

The exact values are learned on the 20% validation split using Macro F₀.₅.

### Important

A weak candidate can be rejected.

An empty prediction is valid.

In [ ]:
# ============================================================
# Cell 20 — Validation feature generation
# ============================================================

validation_feature_paths = {}

for source_num in [2, 3]:
    feature_path = os.path.join(
        artifact_dir,
        f"validation_features_source{source_num}.parquet"
    )

    build_features_from_pairs(
        candidate_paths[("val20", source_num)],
        normalized_paths["train_source1"],
        normalized_paths[f"train_source{source_num}"],
        embedding_paths.get("train_source1"),
        embedding_paths.get(f"train_source{source_num}"),
        feature_path,
        include_label=False,
    )

    validation_feature_paths[source_num] = feature_path
    print(feature_path)


In [ ]:
# ============================================================
# Cell 21 — Validation scoring helpers
# ============================================================

def score_feature_parquet(model, feature_path):
    df = pl.read_parquet(feature_path)
    X = df.select(FEATURE_NAMES).to_numpy().astype(np.float32)

    probabilities = model.predict_proba(X)[:, 1]

    return df.with_columns(
        pl.Series("probability", probabilities.astype(np.float32))
    )

def truth_sets_for_split(source_num, split_name):
    valid_ids = set(
        source1_split
        .filter(pl.col("split") == split_name)
        .get_column("entity_id")
        .to_list()
    )

    truth = defaultdict(set)

    for row in ground_truth_long.filter(
        pl.col("target_source") == f"source{source_num}"
    ).iter_rows(named=True):
        if row["source1_entity_id"] in valid_ids:
            truth[row["source1_entity_id"]].add(row["target_entity_id"])

    for entity_id in valid_ids:
        truth.setdefault(entity_id, set())

    return truth

def prediction_sets(
    scored_df,
    s1_parquet,
    target_parquet,
    threshold,
    semantic_delta,
    margin,
    max_matches,
    valid_source1_ids,
):
    con = duckdb.connect()
    con.register("scored_df", scored_df.to_arrow())

    resolved = con.execute(
        """
        SELECT
            p.*,
            s.entity_id AS s1_id,
            t.entity_id AS target_id
        FROM scored_df p
        JOIN read_parquet(?) s
          ON s.row_idx = p.s1_row_idx
        JOIN read_parquet(?) t
          ON t.row_idx = p.target_row_idx
        """,
        [s1_parquet, target_parquet],
    ).pl()

    con.close()

    result = {x: set() for x in valid_source1_ids}

    for gid, group in resolved.group_by("s1_id", maintain_order=True):
        s1_id = gid[0] if isinstance(gid, tuple) else gid

        if s1_id not in result:
            continue

        group = group.sort("probability", descending=True)

        accepted = []

        for r in group.iter_rows(named=True):
            p = float(r["probability"])

            # Candidates that have no lexical-block evidence rely only on ANN.
            required = (
                threshold
                if int(r["block_hits"]) > 0
                else min(0.995, threshold + semantic_delta)
            )

            if p >= required:
                accepted.append(r)

        # Ambiguity shield:
        # if there is only one accepted candidate but the next best candidate
        # is almost equally probable, abstain.
        if len(accepted) == 1 and len(group) >= 2:
            top1 = float(group[0, "probability"])
            top2 = float(group[1, "probability"])

            if (top1 - top2) < margin:
                accepted = []

        result[s1_id] = {
            str(r["target_id"])
            for r in accepted[:max_matches]
        }

    return result

def f05(true_set, pred_set):
    true_set = set(true_set)
    pred_set = set(pred_set)

    if not true_set:
        return 1.0 if not pred_set else 0.0

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    denom = 0.25 * precision + recall
    if denom == 0:
        return 0.0

    return (1.25 * precision * recall) / denom

def macro_f05(truth, pred):
    values = [
        f05(truth[s1], pred.get(s1, set()))
        for s1 in truth
    ]
    return float(np.mean(values)) if values else 1.0


## 19. Tune the precision shield on the 20% validation set

### Why this matters

Because F₀.₅ is precision-heavy and false merges are especially costly, this stage is as important as the classifier itself.

We search over:

```text
threshold
semantic-only delta
ambiguity margin
```

and choose the combination with the highest **validation Macro F₀.₅**.

In [ ]:
# ============================================================
# Cell 22 — Validation threshold / shield optimization
# ============================================================

shield = {}
max_match_caps = {}

for source_num in [2, 3]:
    model = models[source_num]

    scored_val = score_feature_parquet(
        model,
        validation_feature_paths[source_num]
    )

    truth = truth_sets_for_split(source_num, "val20")

    # Conservative cardinality cap from observed training ground truth.
    source_truth_counts = (
        ground_truth_long
        .filter(pl.col("target_source") == f"source{source_num}")
        .group_by("source1_entity_id")
        .len()["len"]
        .to_numpy()
    )

    if len(source_truth_counts):
        max_cap = max(
            1,
            min(
                20,
                int(np.ceil(np.percentile(source_truth_counts, 99))) + 1
            )
        )
    else:
        max_cap = 5

    max_match_caps[source_num] = max_cap

    best = None

    thresholds = np.arange(0.55, 0.991, 0.02)
    semantic_deltas = [0.00, 0.02, 0.04, 0.06, 0.08]
    margins = [0.00, 0.01, 0.02, 0.03, 0.05, 0.07]

    valid_ids = set(truth.keys())

    for threshold in thresholds:
        for delta in semantic_deltas:
            for margin in margins:

                pred = prediction_sets(
                    scored_val,
                    normalized_paths["train_source1"],
                    normalized_paths[f"train_source{source_num}"],
                    threshold=float(threshold),
                    semantic_delta=float(delta),
                    margin=float(margin),
                    max_matches=max_cap,
                    valid_source1_ids=valid_ids,
                )

                score = macro_f05(truth, pred)

                if best is None or score > best["score"]:
                    best = {
                        "score": float(score),
                        "threshold": float(threshold),
                        "semantic_delta": float(delta),
                        "margin": float(margin),
                    }

    shield[source_num] = best

    print(
        f"Source {source_num}: "
        f"MacroF0.5={best['score']:.6f}, "
        f"threshold={best['threshold']:.3f}, "
        f"semantic_delta={best['semantic_delta']:.3f}, "
        f"margin={best['margin']:.3f}, "
        f"max_matches={max_cap}"
    )

with open(os.path.join(artifact_dir, "shield_config.json"), "w") as f:
    json.dump(shield, f, indent=2)


## 20. Evaluate the untouched 15% test split

### Protocol

Now the shield is frozen.

We use the 15% held-out Source-1 entities exactly once for internal evaluation.

No threshold tuning happens here.

In [ ]:
# ============================================================
# Cell 23 — Held-out 15% evaluation
# ============================================================

heldout_scores = {}

for source_num in [2, 3]:
    # Test15 candidates.
    test15_features = os.path.join(
        artifact_dir,
        f"test15_features_source{source_num}.parquet"
    )

    build_features_from_pairs(
        candidate_paths[("test15", source_num)],
        normalized_paths["train_source1"],
        normalized_paths[f"train_source{source_num}"],
        embedding_paths.get("train_source1"),
        embedding_paths.get(f"train_source{source_num}"),
        test15_features,
        include_label=False,
    )

    scored_test15 = score_feature_parquet(
        models[source_num],
        test15_features
    )

    truth = truth_sets_for_split(source_num, "test15")

    params = shield[source_num]

    pred = prediction_sets(
        scored_test15,
        normalized_paths["train_source1"],
        normalized_paths[f"train_source{source_num}"],
        threshold=params["threshold"],
        semantic_delta=params["semantic_delta"],
        margin=params["margin"],
        max_matches=max_match_caps[source_num],
        valid_source1_ids=set(truth.keys()),
    )

    score = macro_f05(truth, pred)
    heldout_scores[source_num] = score

    print(
        f"HELD-OUT TEST15 Source {source_num} Macro F0.5 = {score:.6f}"
    )

print("Held-out scores:", heldout_scores)


## 21. Refit on 85% labeled entities

### Final training rule

The 15% test set remains untouched.

After threshold/decision rules have been selected on the 20% validation set:

```text
65% train + 20% validation = 85%
```

is used to fit the final pair models.

The shield parameters remain frozen.

In [ ]:
# ============================================================
# Cell 24 — Refit final models on train65 + val20
# ============================================================

# Build labeled validation pairs first.
labeled_85_paths = {}

for source_num in [2, 3]:
    val_labeled = os.path.join(
        artifact_dir,
        f"labeled_val_source{source_num}.parquet"
    )

    if not os.path.exists(val_labeled):
        val_labeled_df = build_labeled_pairs(
            candidate_paths[("val20", source_num)],
            normalized_paths["train_source1"],
            normalized_paths[f"train_source{source_num}"],
            f"source{source_num}",
            "val20",
            max_pairs=MAX_TRAIN_PAIRS_PER_SOURCE,
        )
        val_labeled_df.write_parquet(
            val_labeled,
            compression="zstd"
        )

    labeled_85_paths[source_num] = val_labeled

final_models = {}

for source_num in [2, 3]:
    train_combined = os.path.join(
        artifact_dir,
        f"final_training_pairs_source{source_num}.parquet"
    )

    con = duckdb.connect()

    con.execute(
        """
        COPY (
            SELECT * FROM read_parquet(?)
            UNION ALL
            SELECT * FROM read_parquet(?)
        )
        TO ?
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """,
        [
            labeled_pair_paths[source_num],
            labeled_85_paths[source_num],
            train_combined,
        ],
    )

    con.close()

    train_features_85 = os.path.join(
        artifact_dir,
        f"final_training_features_source{source_num}.parquet"
    )

    build_features_from_pairs(
        train_combined,
        normalized_paths["train_source1"],
        normalized_paths[f"train_source{source_num}"],
        embedding_paths.get("train_source1"),
        embedding_paths.get(f"train_source{source_num}"),
        train_features_85,
        include_label=True,
    )

    model_path = os.path.join(
        artifact_dir,
        f"final_pair_model_source{source_num}.joblib"
    )

    final_models[source_num] = train_pair_classifier(
        train_features_85,
        model_path,
    )

    print("Final model saved:", model_path)


## 22. Official test candidate generation

### Required output #1

Now switch to:

```text
test_source1.tsv
test_source2.tsv
test_source3.tsv
```

The exact union of lexical + ANN candidates is generated.

This candidate set is persisted as:

```text
output/candidate_pairs.tsv
```

The final inference model will score **only this set**.

In [ ]:
# ============================================================
# Cell 25 — Official test candidate generation
# ============================================================

official_candidate_parquets = {}

for source_num in [2, 3]:
    out_path = os.path.join(
        artifact_dir,
        f"official_candidates_source{source_num}.parquet"
    )

    official_candidate_parquets[source_num] = generate_candidate_parquet(
        s1_parquet=normalized_paths["test_source1"],
        target_parquet=normalized_paths[f"test_source{source_num}"],
        target_block_db=test_block_db[source_num],
        target_embedding_path=embedding_paths.get(f"test_source{source_num}"),
        target_faiss_path=faiss_paths.get(f"test_source{source_num}"),
        s1_embedding_path=embedding_paths.get("test_source1"),
        output_parquet=out_path,
        s1_filter=None,
    )

print("Official candidate generation complete.")


## 23. Write the required `candidate_pairs.tsv`

### Audit rule

This TSV is the exact union of candidates immediately before final inference.

Every final predicted match must be contained in this file.

In [ ]:
# ============================================================
# Cell 26 — Required output/candidate_pairs.tsv
# ============================================================

candidate_pairs_file = os.path.join(
    output_dir,
    "candidate_pairs.tsv"
)

os.makedirs(output_dir, exist_ok=True)

# Write Source-2 and Source-3 candidates to one exact auditable TSV.
con = duckdb.connect()

con.execute(
    """
    COPY (
        SELECT
            s.entity_id AS source1_entity_id,
            'source2' AS target_source,
            t.entity_id AS candidate_entity_id,
            c.block_hits,
            c.retrieval_score
        FROM read_parquet(?) c
        JOIN read_parquet(?) s
          ON s.row_idx = c.s1_row_idx
        JOIN read_parquet(?) t
          ON t.row_idx = c.target_row_idx

        UNION ALL

        SELECT
            s.entity_id AS source1_entity_id,
            'source3' AS target_source,
            t.entity_id AS candidate_entity_id,
            c.block_hits,
            c.retrieval_score
        FROM read_parquet(?) c
        JOIN read_parquet(?) s
          ON s.row_idx = c.s1_row_idx
        JOIN read_parquet(?) t
          ON t.row_idx = c.target_row_idx

        ORDER BY source1_entity_id, target_source, candidate_entity_id
    )
    TO ?
    (HEADER TRUE, DELIMITER '\t')
    """,
    [
        official_candidate_parquets[2],
        normalized_paths["test_source1"],
        normalized_paths["test_source2"],
        official_candidate_parquets[3],
        normalized_paths["test_source1"],
        normalized_paths["test_source3"],
        candidate_pairs_file,
    ],
)

con.close()

print("Created:", candidate_pairs_file)
display(
    pl.read_csv(
        candidate_pairs_file,
        separator="\t",
        infer_schema=False,
        n_rows=20,
    )
)


## 24. Final inference feature generation

### No new candidates are created here

This stage is important:

```text
official candidate_pairs
        ↓
feature extraction
        ↓
final model
```

The model cannot introduce a candidate that wasn't present in the saved candidate set.

In [ ]:
# ============================================================
# Cell 27 — Score exact candidate sets with final models
# ============================================================

official_feature_paths = {}
official_scored = {}

for source_num in [2, 3]:

    feature_path = os.path.join(
        artifact_dir,
        f"official_features_source{source_num}.parquet"
    )

    build_features_from_pairs(
        official_candidate_parquets[source_num],
        normalized_paths["test_source1"],
        normalized_paths[f"test_source{source_num}"],
        embedding_paths.get("test_source1"),
        embedding_paths.get(f"test_source{source_num}"),
        feature_path,
        include_label=False,
    )

    official_feature_paths[source_num] = feature_path

    official_scored[source_num] = score_feature_parquet(
        final_models[source_num],
        feature_path
    )

    print(
        f"Source {source_num} scored candidates:",
        len(official_scored[source_num])
    )


## 25. Final matching decision

### Conservative decision logic

For every Source-1 entity:

1. apply the frozen validation threshold;
2. use the stricter threshold for semantic-only candidates;
3. apply the ambiguity margin;
4. enforce the training-derived safety cap;
5. deduplicate entity IDs;
6. allow an empty list.

This is the precision shield.

In [ ]:
# ============================================================
# Cell 28 — Build final matching results
# ============================================================

test_s1_ids = (
    pl.read_parquet(
        normalized_paths["test_source1"],
        columns=["entity_id"]
    )["entity_id"]
    .to_list()
)

fake_test_split = pl.DataFrame({
    "entity_id": test_s1_ids,
    "split": ["official_test"] * len(test_s1_ids),
})

def official_prediction_sets(
    scored_df,
    source_num,
):
    params = shield[source_num]

    return prediction_sets(
        scored_df,
        normalized_paths["test_source1"],
        normalized_paths[f"test_source{source_num}"],
        threshold=params["threshold"],
        semantic_delta=params["semantic_delta"],
        margin=params["margin"],
        max_matches=max_match_caps[source_num],
        valid_source1_ids=set(test_s1_ids),
    )

final_pred_source2 = official_prediction_sets(
    official_scored[2],
    2
)

final_pred_source3 = official_prediction_sets(
    official_scored[3],
    3
)

result_rows = []

for s1_id in test_s1_ids:
    s2_matches = sorted(
        set(final_pred_source2.get(s1_id, set()))
    )
    s3_matches = sorted(
        set(final_pred_source3.get(s1_id, set()))
    )

    result_rows.append({
        "entity_id": s1_id,
        "source2_matches": json.dumps(
            s2_matches,
            ensure_ascii=False,
        ),
        "source3_matches": json.dumps(
            s3_matches,
            ensure_ascii=False,
        ),
    })

matching_results = pl.DataFrame(result_rows)

matching_results_file = os.path.join(
    output_dir,
    "matching_results.tsv"
)

matching_results.write_csv(
    matching_results_file,
    separator="\t",
)

print("Created:", matching_results_file)
display(matching_results.head(20))


## 26. Output integrity checks

### Required checks

We verify:

- one row per Source-1 test entity;
- no duplicate IDs in a prediction list;
- every predicted candidate exists in `candidate_pairs.tsv`;
- no unexpected target source;
- output files exist.

In [ ]:
# ============================================================
# Cell 29 — Required output integrity checks
# ============================================================

candidate_df = pl.read_csv(
    candidate_pairs_file,
    separator="\t",
    infer_schema=False,
)

# Candidate lookup.
candidate_lookup = defaultdict(set)

for r in candidate_df.iter_rows(named=True):
    candidate_lookup[
        (r["source1_entity_id"], r["target_source"])
    ].add(r["candidate_entity_id"])

# Check one row per Source-1.
assert matching_results.height == len(test_s1_ids)
assert matching_results["entity_id"].n_unique() == len(test_s1_ids)

# Validate lists and candidate containment.
violations = []

for r in matching_results.iter_rows(named=True):
    s1 = r["entity_id"]

    for source_name, col in [
        ("source2", "source2_matches"),
        ("source3", "source3_matches"),
    ]:
        pred = parse_id_list(r[col])

        if len(pred) != len(set(pred)):
            violations.append(
                (s1, source_name, "duplicate_entity_id")
            )

        allowed = candidate_lookup[(s1, source_name)]

        for target_id in pred:
            if target_id not in allowed:
                violations.append(
                    (s1, source_name, f"not_in_candidate_set:{target_id}")
                )

print("Violation count:", len(violations))

if violations:
    print("First violations:")
    for v in violations[:20]:
        print(v)
    raise AssertionError("Output integrity check failed.")

print("✅ Candidate containment check passed.")
print("✅ Duplicate prediction check passed.")
print("✅ One row per Source-1 entity passed.")


## 27. Validate the submission with the supplied validator

### Important

The local challenge validator is the final authority on:

- exact output column names;
- array/list formatting;
- required number of rows;
- other submission-format rules.

The folder tree supplied with the challenge contains:

```text
student_resource/utils/validate_submission.py
```

Run it before submission.

In [ ]:
# ============================================================
# Cell 30 — Run supplied validator
# ============================================================

print("Validator file:", validator_file)
print("Submission file:", matching_results_file)

if os.path.exists(validator_file):
    print("\nFirst part of validator:\n")
    with open(validator_file, "r", encoding="utf-8", errors="replace") as f:
        validator_text = f.read()

    print(validator_text[:5000])

    print("\n" + "=" * 70)
    print("Run the validator from Anaconda Prompt:")
    print()
    print(f'python "{validator_file}" "{matching_results_file}"')
else:
    print("Validator file not found. Check utils_dir.")


## 28. Final submission package

### Expected output structure

After the notebook finishes:

```text
student_resource/
├── output/
│   ├── candidate_pairs.tsv
│   └── matching_results.tsv
│
├── artifacts/
│   ├── normalized Parquet
│   ├── block indexes
│   ├── embeddings
│   ├── FAISS indexes
│   ├── features
│   ├── models
│   └── shield_config.json
│
├── utils/
│   └── validate_submission.py
│
└── dataset/
    ├── train/
    └── test/
```

### What each required file means

`candidate_pairs.tsv`

> Exact final candidate set considered by the inference model.

`matching_results.tsv`

> Final submission predictions after the precision shield.

Do not manually add matches to `matching_results.tsv` that are absent from `candidate_pairs.tsv`.

### Final rule

Do not tune thresholds on the official hidden test data.

The notebook tunes the decision layer on the 20% validation split and reports the 15% held-out split before official inference.